In [19]:
import math

class Sommet:
    def __init__(self, id, label):
        self.id = id
        self.label = label
        self.voisins = []

    def ajout_voisin(self, nouveau_voisin):
        if nouveau_voisin not in self.voisins:
            self.voisins.append(nouveau_voisin)

class Graphe:
    def __init__(self):
        self.sommets = {}
        self.n = 0
        self.m = 0

    def nombre_sommets(self):
        return self.n

    def nombre_aretes(self):
        return self.m
        
    def ajout_sommet(self, classe_sommet, label = ""):
        nouveau_sommet = classe_sommet(self.n, label)
        self.sommets[self.n] = nouveau_sommet
        self.n += 1

    def ajout_arete(self, id_sommet_1, id_sommet_2):
        self.sommets[id_sommet_1].ajout_voisin(id_sommet_2)
        self.sommets[id_sommet_2].ajout_voisin(id_sommet_1)
        self.m += 1

    def ajout_arete_digraphe(self, id_sommet_1, id_sommet_2):
        self.sommets[id_sommet_1].ajout_voisin(id_sommet_2)
        self.m += 1
    
    def voisinage_sommet(self, id_sommet):
        return self.sommets[id_sommet].voisins

    def __str__(self):
        if self.n == 0:
            return "Le graphe est vide."
        
        resultat = "Le graphe est composé des sommets :\n"
        
        for id in self.sommets:
            s = self.sommets[id]
            resultat += "-\tNuméro " + str(s.id)
            if s.label != "":
                resultat += " (label: " + s.label + ")"
            resultat += ", ayant pour voisin(s) le(s) sommet(s) numéro(s) : " + str(s.voisins) + "\n"
        resultat += "Au total, " + str(self.n) + " sommets et " + str(self.m) + " arêtes."
        return resultat
    
    def __getitem__(self, id):
        return self.sommets[id]

class SommetDFS(Sommet):
    def __init__(self, id, label):
        Sommet.__init__(self, id, label)
        self.couleur = "blanc"
        self.distance = math.inf 
        self.parent = None
        self.d = -1
        self.f = -1

    def initialisation_dfs(self):
        self.couleur = "blanc"
        self.distance = math.inf
        self.parent = None
        self.d = -1
        self.f = -1 

In [20]:
def parcours_en_profondeur_sommet(graphe, id_sommet, temps, nb_iterations, ordre_topologique, types_arcs):
    nb_iterations += 1
    print("Itération " + str(nb_iterations) + ". Sommet numéro " + str(id_sommet) + " exploré.")
    graphe[id_sommet].couleur = "gris"
    graphe[id_sommet].d = temps
    temps += 1

    for id_voisin in graphe.voisinage_sommet(id_sommet):
        voisin = graphe[id_voisin]
        if voisin.couleur == "blanc":
            types_arcs[(id_sommet, id_voisin)] = "foret"
            voisin.parent = id_sommet
            temps, nb_iterations, ordre_topologique, types_arcs = parcours_en_profondeur_sommet(graphe, 
                                                                                                id_voisin, 
                                                                                                temps, 
                                                                                                nb_iterations, 
                                                                                                ordre_topologique, 
                                                                                                types_arcs)
        elif voisin.couleur == "gris":
            types_arcs[(id_sommet, id_voisin)] = "retour"
        else: # Forcément de couleur noire
            if graphe[id_sommet].d < voisin.d:
                types_arcs[(id_sommet, id_voisin)] = "avant"
            else: # On ne peut avoir d'égalité de temps de toutes façons
                types_arcs[(id_sommet, id_voisin)] = "traversant"
    
    temps += 1
    graphe[id_sommet].couleur = "noir"
    graphe[id_sommet].f = temps

    ordre_topologique.append(id_sommet)
    
    return temps, nb_iterations, ordre_topologique, types_arcs

In [28]:
def ordre_topologique_et_types_arcs(graphe):
    for id_sommet in graphe.sommets:
        graphe[id_sommet].initialisation_dfs()

    ordre_topologique = []
    types_arcs = {} 
    
    temps = 0
    nb_iterations = 0

    for id_sommet in graphe.sommets:
        if graphe[id_sommet].couleur == "blanc":
            print("Départ", id_sommet)
            temps, nb_iterations, ordre_topologique, types_arcs = parcours_en_profondeur_sommet(graphe, 
                                                                                                id_sommet, 
                                                                                                temps, 
                                                                                                nb_iterations, 
                                                                                                ordre_topologique,
                                                                                                types_arcs)

    # L'ordre topologique est l'inverse de l'ordre de fin DFS
    ordre_topologique.reverse()
    return ordre_topologique, types_arcs

In [29]:
g = Graphe()
for _ in range(6):
    g.ajout_sommet(SommetDFS)

g.ajout_arete_digraphe(0, 1)
g.ajout_arete_digraphe(0, 2)
g.ajout_arete_digraphe(1, 3)
g.ajout_arete_digraphe(2, 3)
g.ajout_arete_digraphe(3, 4)
g.ajout_arete_digraphe(4, 5)

print(g)

Le graphe est composé des sommets :
-	Numéro 0, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [1, 2]
-	Numéro 1, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [3]
-	Numéro 2, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [3]
-	Numéro 3, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [4]
-	Numéro 4, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [5]
-	Numéro 5, ayant pour voisin(s) le(s) sommet(s) numéro(s) : []
Au total, 6 sommets et 6 arêtes.


In [30]:
ordre, types_arcs = ordre_topologique_et_types_arcs(g)
print("Ordre topologique :", ordre)
print("Type des arcs:", types_arcs)

Départ 0
Itération 1. Sommet numéro 0 exploré.
Itération 2. Sommet numéro 1 exploré.
Itération 3. Sommet numéro 3 exploré.
Itération 4. Sommet numéro 4 exploré.
Itération 5. Sommet numéro 5 exploré.
Itération 6. Sommet numéro 2 exploré.
Ordre topologique : [0, 2, 1, 3, 4, 5]
Type des arcs: {(0, 1): 'foret', (1, 3): 'foret', (3, 4): 'foret', (4, 5): 'foret', (0, 2): 'foret', (2, 3): 'traversant'}


In [31]:
g = Graphe()
for _ in range(16):
    g.ajout_sommet(SommetDFS)

g.ajout_arete_digraphe(0, 1)
g.ajout_arete_digraphe(1, 5)
g.ajout_arete_digraphe(5, 6)
g.ajout_arete_digraphe(6, 0)
g.ajout_arete_digraphe(4, 5)
g.ajout_arete_digraphe(6, 2)
g.ajout_arete_digraphe(3, 2)
g.ajout_arete_digraphe(2, 7)
g.ajout_arete_digraphe(7, 3)
g.ajout_arete_digraphe(6, 2)
g.ajout_arete_digraphe(4, 8)
g.ajout_arete_digraphe(5, 11)
g.ajout_arete_digraphe(6, 10)
g.ajout_arete_digraphe(10, 7)
g.ajout_arete_digraphe(10, 11)
g.ajout_arete_digraphe(9, 10)
g.ajout_arete_digraphe(8, 13)
g.ajout_arete_digraphe(9, 12)
g.ajout_arete_digraphe(13, 9)
g.ajout_arete_digraphe(12, 8)
g.ajout_arete_digraphe(13, 14)
g.ajout_arete_digraphe(14, 10)
g.ajout_arete_digraphe(11, 14)
g.ajout_arete_digraphe(11, 15)

print(g)

Le graphe est composé des sommets :
-	Numéro 0, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [1]
-	Numéro 1, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [5]
-	Numéro 2, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [7]
-	Numéro 3, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [2]
-	Numéro 4, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [5, 8]
-	Numéro 5, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [6, 11]
-	Numéro 6, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [0, 2, 10]
-	Numéro 7, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [3]
-	Numéro 8, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [13]
-	Numéro 9, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [10, 12]
-	Numéro 10, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [7, 11]
-	Numéro 11, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [14, 15]
-	Numéro 12, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [8]
-	Numéro 13, ayant pour voisin(s) le(s) sommet(s) numéro(s) : [9, 14]
-	Numéro 14, ayan

In [32]:
ordre, types_arcs = ordre_topologique_et_types_arcs(g)
print("Ordre topologique :", ordre)
print("Type des arcs:", types_arcs)

Départ 0
Itération 1. Sommet numéro 0 exploré.
Itération 2. Sommet numéro 1 exploré.
Itération 3. Sommet numéro 5 exploré.
Itération 4. Sommet numéro 6 exploré.
Itération 5. Sommet numéro 2 exploré.
Itération 6. Sommet numéro 7 exploré.
Itération 7. Sommet numéro 3 exploré.
Itération 8. Sommet numéro 10 exploré.
Itération 9. Sommet numéro 11 exploré.
Itération 10. Sommet numéro 14 exploré.
Itération 11. Sommet numéro 15 exploré.
Départ 4
Itération 12. Sommet numéro 4 exploré.
Itération 13. Sommet numéro 8 exploré.
Itération 14. Sommet numéro 13 exploré.
Itération 15. Sommet numéro 9 exploré.
Itération 16. Sommet numéro 12 exploré.
Ordre topologique : [4, 8, 13, 9, 12, 0, 1, 5, 6, 10, 11, 15, 14, 2, 7, 3]
Type des arcs: {(0, 1): 'foret', (1, 5): 'foret', (5, 6): 'foret', (6, 0): 'retour', (6, 2): 'foret', (2, 7): 'foret', (7, 3): 'foret', (3, 2): 'retour', (6, 10): 'foret', (10, 7): 'traversant', (10, 11): 'foret', (11, 14): 'foret', (14, 10): 'retour', (11, 15): 'foret', (5, 11): 'avan